# 02 — Prepare and clean the LLM response data

**Purpose:** convert `data_collection_all_responses.xlsx` from five wide domain sheets into a transparent, long-format response dataset that is safe to use in later dissertation notebooks.

This notebook:

- never overwrites or edits the collected workbook;
- keeps one audit row for every prompt–model combination;
- standardises names, data types, timestamps and whitespace;
- removes empty/manual collection fields only from the compact analysis view;
- creates objective text and structure fields useful for later analysis;
- detects missing responses, duplicate keys and responses that reached the collection notebook's 1,200-token ceiling;
- records missing and token-limited responses as collection limitations without altering their text;
- always exports the cleaned table as `analysis_ready_responses.csv`.

It does **not** invent human hallucination ratings, entities, brands or evaluation notes when those source fields are blank. Semantic similarity, recommendation drift and inferential testing belong in the later analysis notebooks.


## Step 1 — Imports and reproducibility settings

Google Colab normally already contains these packages. The first code cell only installs `openpyxl` if it is genuinely missing.


In [ ]:
import importlib.util
import json
import os
import re
import subprocess
import sys
import unicodedata
import zipfile
from pathlib import Path

if importlib.util.find_spec("openpyxl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl>=3.1,<4"])

import numpy as np
import pandas as pd
import openpyxl

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("openpyxl:", openpyxl.__version__)


## Step 2 — Find the collected workbook

Recommended Colab use: place `data_collection_all_responses.xlsx` directly in **My Drive**, then run this cell. If it is not found, Colab opens an upload box. Advanced users can override the paths with the `DATA_COLLECTION_FILE` and `CLEAN_OUTPUT_DIR` environment variables.


In [ ]:
IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = True
FILE_NAME = "data_collection_all_responses.xlsx"

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

override = os.environ.get("DATA_COLLECTION_FILE", "").strip()
candidates = [
    Path(override) if override else None,
    Path("/content/drive/MyDrive") / FILE_NAME,
    Path("/content") / FILE_NAME,
    Path.cwd() / FILE_NAME,
]
INPUT_FILE = next((p for p in candidates if p is not None and p.exists()), None)

if INPUT_FILE is None and IN_COLAB:
    from google.colab import files
    print(f"Please upload {FILE_NAME}")
    uploaded = files.upload()
    if FILE_NAME not in uploaded:
        raise FileNotFoundError(f"The uploaded file must be named {FILE_NAME}")
    INPUT_FILE = Path("/content") / FILE_NAME

if INPUT_FILE is None:
    raise FileNotFoundError(
        f"Could not find {FILE_NAME}. Put it in My Drive, upload it to Colab, "
        "or set DATA_COLLECTION_FILE to its full path."
    )

output_override = os.environ.get("CLEAN_OUTPUT_DIR", "").strip()
if output_override:
    OUTPUT_DIR = Path(output_override)
elif IN_COLAB and USE_GOOGLE_DRIVE:
    OUTPUT_DIR = Path("/content/drive/MyDrive/dissertation_clean_data")
else:
    OUTPUT_DIR = Path.cwd() / "dissertation_clean_data"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input:", INPUT_FILE)
print("Output folder:", OUTPUT_DIR)


## Step 3 — Define the experimental schema

The expected design is fixed: five domains, 100 base intents, 10 semantically equivalent phrasings per base intent, 1,000 prompts and three model responses per prompt.


In [ ]:
DOMAIN_SHEETS = ["Technology", "Finance", "Healthcare", "Marketing", "Sports"]
MODELS = ["ChatGPT", "Claude", "Gemini"]

EXPECTED_PROMPT_ROWS = 1_000
EXPECTED_BASE_INTENTS = 100
EXPECTED_PHRASINGS = 10
EXPECTED_LONG_ROWS = 3_000
EXPECTED_GROUP_SIZE = 10

# This value matches MAX_OUTPUT_TOKENS in Notebook 01.
COLLECTION_MAX_OUTPUT_TOKENS = 1_200

BASE_COLUMN_MAP = {
    "Prompt ID": "prompt_id",
    "Domain": "domain",
    "Base Intent ID": "base_intent_id",
    "Variation No": "variation_no",
    "Intent Type": "intent_type",
    "Phrasing Type": "phrasing_type",
    "Prompt": "prompt_text",
    "Run Number": "run_number",
    "Search/Web Enabled": "search_web_enabled",
    "Reference / Verification Source": "reference_verification_source",
    "Collection Priority": "collection_priority",
    "Completion Status": "completion_status",
    "Reviewer": "reviewer",
    "Quality Check Status": "quality_check_status",
    "Data Collection Notes": "data_collection_notes",
    "source_sheet": "source_sheet",
    "source_row_number": "source_row_number",
}

MODEL_SUFFIX_MAP = {
    "Model Version": "model_version",
    "Timestamp": "timestamp",
    "Temperature": "temperature_setting",
    "Input Tokens": "input_tokens",
    "Output Tokens": "output_tokens",
    "Response Text": "response_text",
    "Citation Count": "citation_count_collected",
    "URLs / Sources": "urls_sources_collected",
    "Entities / Brands Mentioned": "entities_brands_collected",
    "Recommendations / Ranked Items": "recommendations_collected",
    "Answer Type": "answer_type_collected",
    "Hallucination Score (0-3)": "hallucination_score_human",
    "Evaluation Notes": "evaluation_notes_collected",
}

MISSING_RESPONSE_MARKERS = {
    "", "na", "n/a", "none", "null", "nan", "missing", "error", "failed",
    "api error", "no response", "no answer", "not available",
}
URL_PATTERN = re.compile(r"https?://[^\s<>()\[\]{}\"']+", re.I)
WORD_PATTERN = re.compile(r"\b[A-Za-z0-9]+(?:['’\-][A-Za-z0-9]+)*\b")


## Step 4 — Load and validate the five domain sheets

The `Dashboard` sheet is intentionally excluded because it is a presentation summary, not observation-level research data.


In [ ]:
excel_file = pd.ExcelFile(INPUT_FILE)
missing_sheets = [sheet for sheet in DOMAIN_SHEETS if sheet not in excel_file.sheet_names]
if missing_sheets:
    raise ValueError(f"Required domain sheets are missing: {missing_sheets}")

wide_frames = []
for sheet_name in DOMAIN_SHEETS:
    frame = pd.read_excel(INPUT_FILE, sheet_name=sheet_name)
    frame = frame.dropna(how="all").copy()
    frame["source_sheet"] = sheet_name
    frame["source_row_number"] = np.arange(2, len(frame) + 2)
    wide_frames.append(frame)

wide = pd.concat(wide_frames, ignore_index=True)

required_base = [column for column in BASE_COLUMN_MAP if column not in {"source_sheet", "source_row_number"}]
required_model = [f"{model} {suffix}" for model in MODELS for suffix in MODEL_SUFFIX_MAP]
missing_columns = [column for column in required_base + required_model if column not in wide.columns]
if missing_columns:
    raise KeyError(f"The workbook is missing required columns: {missing_columns}")

design_preview = pd.DataFrame({
    "measure": ["prompt rows", "unique prompts", "base intents", "phrasing types", "domain sheets"],
    "observed": [
        len(wide), wide["Prompt ID"].nunique(), wide["Base Intent ID"].nunique(),
        wide["Phrasing Type"].nunique(), wide["source_sheet"].nunique(),
    ],
    "expected": [EXPECTED_PROMPT_ROWS, EXPECTED_PROMPT_ROWS, EXPECTED_BASE_INTENTS, EXPECTED_PHRASINGS, len(DOMAIN_SHEETS)],
})
design_preview["passed"] = design_preview["observed"] == design_preview["expected"]
display(design_preview)

if not design_preview["passed"].all():
    raise ValueError("The experimental design checks failed. Review the table before continuing.")


## Step 5 — Reshape from wide to long format

Each output row represents one model's response to one prompt. This is the correct unit of observation for response-level cleaning and later grouped prompt-sensitivity analysis.


In [ ]:
def clean_single_line(value) -> str:
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).replace("\u00a0", " ")
    return re.sub(r"\s+", " ", text).strip()


def clean_multiline(value) -> str:
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).replace("\r\n", "\n").replace("\r", "\n")
    lines = [re.sub(r"[ \t]+", " ", line).rstrip() for line in text.split("\n")]
    result = []
    previous_blank = False
    for line in lines:
        blank = not line.strip()
        if blank and previous_blank:
            continue
        result.append(line.strip() if line.strip() else "")
        previous_blank = blank
    return "\n".join(result).strip()


parts = []
for model_name in MODELS:
    model_map = {f"{model_name} {suffix}": clean_name for suffix, clean_name in MODEL_SUFFIX_MAP.items()}
    selected = list(BASE_COLUMN_MAP) + list(model_map)
    part = wide[selected].rename(columns={**BASE_COLUMN_MAP, **model_map}).copy()
    part["model_name"] = model_name
    parts.append(part)

processed = pd.concat(parts, ignore_index=True)

single_line_columns = [
    "prompt_id", "domain", "base_intent_id", "intent_type", "phrasing_type",
    "reference_verification_source", "collection_priority", "completion_status",
    "reviewer", "quality_check_status", "data_collection_notes", "source_sheet",
    "model_version", "temperature_setting", "urls_sources_collected",
    "entities_brands_collected", "recommendations_collected", "answer_type_collected",
    "evaluation_notes_collected",
]
for column in single_line_columns:
    processed[column] = processed[column].map(clean_single_line)

processed["prompt_text"] = processed["prompt_text"].map(clean_multiline)
processed["response_text"] = processed["response_text"].map(clean_multiline)

for column in ["variation_no", "run_number", "input_tokens", "output_tokens", "citation_count_collected"]:
    processed[column] = pd.to_numeric(processed[column], errors="coerce").astype("Int64")

processed["hallucination_score_human"] = pd.to_numeric(
    processed["hallucination_score_human"], errors="coerce"
).astype("Float64")
processed["timestamp"] = pd.to_datetime(processed["timestamp"], errors="coerce", utc=True)
processed["timestamp"] = processed["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ").fillna("")
processed["search_web_enabled"] = (
    processed["search_web_enabled"].astype(str).str.strip().str.lower().isin({"true", "yes", "1", "enabled", "on"})
)

processed["response_id"] = (
    processed["prompt_id"] + "__" + processed["model_name"].str.lower() + "__run" + processed["run_number"].fillna(0).astype(int).astype(str)
)
processed["group_id"] = (
    processed["domain"] + "__" + processed["base_intent_id"] + "__" + processed["model_name"].str.lower()
)

print("Long-format shape:", processed.shape)
display(processed[["response_id", "domain", "base_intent_id", "phrasing_type", "model_name", "response_text"]].head(6))


## Step 6 — Create objective text features and quality flags

Reaching exactly 1,200 output tokens is retained as a collection-limitation flag because the collection notebook set that value as its maximum. The notebook does not delete, extend or guess missing continuation text.


In [ ]:
def extract_words(text: str):
    return WORD_PATTERN.findall(text)


def sentence_count(text: str) -> int:
    if not text:
        return 0
    return len([part for part in re.split(r"(?<=[.!?])\s+|\n+", text) if part.strip()])


def ends_cleanly(text: str) -> bool:
    if not text:
        return False
    return bool(re.search(r"(?:[.!?][\"'’”)*_`\]}>]*|```|\|)\s*$", text))


processed["response_valid"] = (
    processed["response_text"].ne("")
    & ~processed["response_text"].str.lower().isin(MISSING_RESPONSE_MARKERS)
)
processed["response_character_count"] = processed["response_text"].str.len().astype(int)
processed["response_word_count"] = processed["response_text"].map(lambda text: len(extract_words(text))).astype(int)
processed["response_sentence_count"] = processed["response_text"].map(sentence_count).astype(int)
processed["response_paragraph_count"] = processed["response_text"].map(
    lambda text: len([part for part in re.split(r"\n\s*\n", text) if part.strip()]) if text else 0
).astype(int)
processed["unique_word_count"] = processed["response_text"].map(
    lambda text: len({word.casefold() for word in extract_words(text)})
).astype(int)
processed["lexical_diversity"] = (
    processed["unique_word_count"] / processed["response_word_count"].replace(0, np.nan)
).round(6)

processed["url_count"] = processed["response_text"].map(
    lambda text: len(set(url.rstrip(".,;:!?)]}") for url in URL_PATTERN.findall(text)))
).astype(int)
processed["has_url"] = processed["url_count"].gt(0)
processed["contains_bulleted_list"] = processed["response_text"].str.contains(r"(?m)^\s*[-*•]\s+", regex=True)
processed["contains_numbered_list"] = processed["response_text"].str.contains(r"(?m)^\s*\d+[.)]\s+", regex=True)
processed["contains_heading"] = processed["response_text"].str.contains(r"(?m)^\s{0,3}#{1,6}\s+", regex=True)
processed["contains_table"] = processed["response_text"].str.contains(r"(?m)^\s*\|.+\|\s*$", regex=True)
processed["contains_code"] = processed["response_text"].str.contains("```", regex=False)
processed["contains_recommendation_language"] = processed["response_text"].str.contains(
    r"\b(?:recommend|suggest|best|top choice|consider)\w*\b", case=False, regex=True
)
processed["contains_refusal_language"] = processed["response_text"].str.contains(
    r"\b(?:i cannot|i can't|unable to|cannot assist|won't help)\b", case=False, regex=True
)
processed["contains_uncertainty_language"] = processed["response_text"].str.contains(
    r"\b(?:may|might|could|possibly|likely|uncertain|depends|not sure)\b", case=False, regex=True
)
processed["contains_numeric_claim"] = processed["response_text"].str.contains(r"\b\d+(?:\.\d+)?%?\b", regex=True)

processed["response_ends_cleanly"] = processed["response_text"].map(ends_cleanly)
processed["at_output_token_limit"] = processed["output_tokens"].eq(COLLECTION_MAX_OUTPUT_TOKENS).fillna(False)
processed["possible_abrupt_truncation"] = (
    processed["response_valid"] & processed["at_output_token_limit"] & ~processed["response_ends_cleanly"]
)

key_columns = ["prompt_id", "model_name", "run_number"]
processed["duplicate_prompt_model_run"] = processed.duplicated(key_columns, keep=False)
processed["duplicate_response_text"] = (
    processed["response_valid"]
    & processed.duplicated(["model_name", "response_text"], keep=False)
)

def collection_limitation(row) -> str:
    issues = []
    if not row.response_valid:
        issues.append("missing_or_invalid_response")
    if row.at_output_token_limit:
        issues.append(f"reached_{COLLECTION_MAX_OUTPUT_TOKENS}_token_limit")
    if row.duplicate_prompt_model_run:
        issues.append("duplicate_prompt_model_run")
    return "|".join(issues)


processed["collection_limitation"] = processed.apply(collection_limitation, axis=1)
processed["has_collection_limitation"] = processed["collection_limitation"].ne("")
processed["analysis_eligible_without_limitation"] = processed["response_valid"] & ~processed["has_collection_limitation"]

qc_by_model = processed.groupby("model_name", observed=True).agg(
    rows=("response_id", "size"),
    valid_responses=("response_valid", "sum"),
    missing_or_invalid=("response_valid", lambda values: int((~values).sum())),
    reached_token_limit=("at_output_token_limit", "sum"),
    abrupt_at_limit=("possible_abrupt_truncation", "sum"),
    flagged_responses=("has_collection_limitation", "sum"),
).reset_index()
display(qc_by_model)


## Step 7 — Validate balance and record the collection limitations

The cleaned data can be analysed with the quality flags retained. The status distinguishes a fully complete collection from an analysis-ready collection with documented limitations; it does not imply that the cleaning failed.


In [ ]:
duplicate_key_count = int(processed.duplicated(["prompt_id", "model_name", "run_number"]).sum())
missing_response_count = int((~processed["response_valid"]).sum())
token_limit_count = int(processed["at_output_token_limit"].sum())
flagged_response_count = int(processed["has_collection_limitation"].sum())

valid_group_sizes = processed.loc[processed["response_valid"]].groupby("group_id").size()
complete_valid_groups = int(valid_group_sizes.eq(EXPECTED_GROUP_SIZE).sum())
total_expected_groups = EXPECTED_BASE_INTENTS * len(MODELS)

checks = pd.DataFrame([
    {"check": "prompt rows", "observed": len(wide), "expected": EXPECTED_PROMPT_ROWS, "passed": len(wide) == EXPECTED_PROMPT_ROWS, "severity": "critical"},
    {"check": "unique prompt IDs", "observed": wide["Prompt ID"].nunique(), "expected": EXPECTED_PROMPT_ROWS, "passed": wide["Prompt ID"].nunique() == EXPECTED_PROMPT_ROWS, "severity": "critical"},
    {"check": "base intents", "observed": wide["Base Intent ID"].nunique(), "expected": EXPECTED_BASE_INTENTS, "passed": wide["Base Intent ID"].nunique() == EXPECTED_BASE_INTENTS, "severity": "critical"},
    {"check": "long-format rows", "observed": len(processed), "expected": EXPECTED_LONG_ROWS, "passed": len(processed) == EXPECTED_LONG_ROWS, "severity": "critical"},
    {"check": "duplicate prompt-model-run keys", "observed": duplicate_key_count, "expected": 0, "passed": duplicate_key_count == 0, "severity": "critical"},
    {"check": "missing or invalid responses", "observed": missing_response_count, "expected": 0, "passed": missing_response_count == 0, "severity": "critical"},
    {"check": "responses at 1,200-token ceiling", "observed": token_limit_count, "expected": 0, "passed": token_limit_count == 0, "severity": "critical"},
    {"check": "complete valid model–intent groups", "observed": complete_valid_groups, "expected": total_expected_groups, "passed": complete_valid_groups == total_expected_groups, "severity": "critical"},
])

collection_complete = bool(checks.loc[checks["severity"].eq("critical"), "passed"].all())
dataset_status = "ANALYSIS_READY_COMPLETE" if collection_complete else "ANALYSIS_READY_WITH_LIMITATIONS"
processed["dataset_status"] = dataset_status

display(checks)
print("Dataset status:", dataset_status)
if not collection_complete:
    print(
        f"Proceed with documented limitations: {flagged_response_count:,} prompt–model rows have a collection flag. "
        "Keep the flags in all analyses and include truncation-aware robustness checks."
    )


## Step 8 — Build the compact analysis view

The full processed audit file keeps collection and provenance fields. The compact analysis view removes blank manual annotation fields and other collection-only columns, but retains the experimental design, model metadata, response text, objective features and all quality flags.


In [ ]:
analysis_columns = [
    "response_id", "group_id", "prompt_id", "base_intent_id", "variation_no", "run_number",
    "domain", "intent_type", "phrasing_type", "prompt_text", "model_name", "model_version",
    "timestamp", "search_web_enabled", "temperature_setting", "input_tokens", "output_tokens",
    "response_text", "response_valid", "response_character_count", "response_word_count",
    "response_sentence_count", "response_paragraph_count", "unique_word_count", "lexical_diversity",
    "url_count", "has_url", "contains_bulleted_list", "contains_numbered_list", "contains_heading",
    "contains_table", "contains_code", "contains_recommendation_language", "contains_refusal_language",
    "contains_uncertainty_language", "contains_numeric_claim", "response_ends_cleanly",
    "at_output_token_limit", "possible_abrupt_truncation", "duplicate_prompt_model_run",
    "duplicate_response_text", "has_collection_limitation", "collection_limitation",
    "analysis_eligible_without_limitation",
    "dataset_status",
]

analysis_ready = processed.loc[
    processed["response_valid"] & ~processed.duplicated(["prompt_id", "model_name", "run_number"], keep="first"),
    analysis_columns,
].copy()
analysis_ready = analysis_ready.sort_values(
    ["domain", "base_intent_id", "variation_no", "prompt_id", "model_name", "run_number"],
    kind="stable",
).reset_index(drop=True)

excluded_invalid = processed.loc[~processed["response_valid"]].copy()
flagged_responses = processed.loc[processed["has_collection_limitation"], [
    "source_sheet", "source_row_number", "prompt_id", "base_intent_id", "variation_no", "domain",
    "intent_type", "phrasing_type", "prompt_text", "model_name", "model_version", "run_number",
    "output_tokens", "response_word_count", "response_ends_cleanly", "response_text",
    "collection_limitation", "data_collection_notes",
]].sort_values(["source_sheet", "source_row_number", "model_name"], kind="stable").reset_index(drop=True)

print("Processed audit rows:", len(processed))
print("Compact valid-response rows:", len(analysis_ready))
print("Excluded invalid-response rows:", len(excluded_invalid))
print("Quality-flagged responses:", len(flagged_responses))
display(flagged_responses.head(12))


## Step 9 — Create a data dictionary and export the reproducible package

CSV is used for the tabular outputs because it is stable, machine-readable and easy to load in Colab, R, SPSS or Excel. The original `.xlsx` remains the immutable raw source.


In [ ]:
descriptions = {
    "response_id": "Unique prompt–model–run identifier.",
    "group_id": "Model-specific base-intent group used to compare the 10 phrasings.",
    "prompt_id": "Unique prompt identifier from the prompt bank.",
    "base_intent_id": "Identifier shared by 10 semantically equivalent phrasings.",
    "variation_no": "Prompt variation number, expected 1–10.",
    "run_number": "Collection run number retained for provenance.",
    "domain": "Prompt domain: Technology, Finance, Healthcare, Marketing or Sports.",
    "intent_type": "Prompt intent category from the experimental design.",
    "phrasing_type": "Linguistic phrasing condition.",
    "prompt_text": "Cleaned prompt text; wording is otherwise unchanged.",
    "model_name": "Normalised provider/model family label.",
    "model_version": "Exact dated or provider-returned model version.",
    "timestamp": "UTC collection timestamp in ISO 8601 format.",
    "search_web_enabled": "Whether web/search access was enabled during collection.",
    "temperature_setting": "Temperature metadata exactly as recorded by collection.",
    "input_tokens": "Provider-reported input token count.",
    "output_tokens": "Provider-reported output token count.",
    "response_text": "Cleaned response text with paragraph structure preserved.",
    "response_valid": "True when usable response text is present.",
    "response_character_count": "Number of characters in cleaned response text.",
    "response_word_count": "Regex-based word count.",
    "response_sentence_count": "Rule-based sentence/line count.",
    "response_paragraph_count": "Count of non-empty paragraph blocks.",
    "unique_word_count": "Case-insensitive count of distinct word tokens.",
    "lexical_diversity": "Unique words divided by total words.",
    "url_count": "Count of distinct visible HTTP/HTTPS URLs in response text.",
    "has_url": "True when at least one visible URL is present.",
    "contains_bulleted_list": "Rule-based indicator for a Markdown bullet list.",
    "contains_numbered_list": "Rule-based indicator for a numbered list.",
    "contains_heading": "Rule-based indicator for a Markdown heading.",
    "contains_table": "Rule-based indicator for a Markdown table row.",
    "contains_code": "Indicator for a fenced code block.",
    "contains_recommendation_language": "Rule-based indicator for recommendation wording.",
    "contains_refusal_language": "Rule-based indicator for refusal wording.",
    "contains_uncertainty_language": "Rule-based indicator for uncertainty wording.",
    "contains_numeric_claim": "Rule-based indicator for a number or percentage.",
    "response_ends_cleanly": "Heuristic terminal-punctuation/structure flag.",
    "at_output_token_limit": "True when output tokens equal the 1,200-token collection ceiling.",
    "possible_abrupt_truncation": "True when a response hit the ceiling and lacks a clean ending.",
    "duplicate_prompt_model_run": "True when the experimental key occurs more than once.",
    "duplicate_response_text": "True when a model produced identical valid text for multiple rows.",
    "has_collection_limitation": "True when the row is missing, duplicated or reached the collection token ceiling.",
    "collection_limitation": "Pipe-separated description of the collection limitation.",
    "analysis_eligible_without_limitation": "True only for valid rows with no collection-limitation flag.",
    "dataset_status": "ANALYSIS_READY_COMPLETE or ANALYSIS_READY_WITH_LIMITATIONS.",
}
data_dictionary = pd.DataFrame([
    {
        "column": column,
        "description": descriptions.get(column, "Audit/provenance field retained from the source workbook."),
        "dtype": str(analysis_ready[column].dtype),
    }
    for column in analysis_columns
])

summary = {
    "input_file": INPUT_FILE.name,
    "dataset_status": dataset_status,
    "wide_prompt_rows": int(len(wide)),
    "unique_prompts": int(wide["Prompt ID"].nunique()),
    "base_intents": int(wide["Base Intent ID"].nunique()),
    "long_prompt_model_rows": int(len(processed)),
    "valid_response_rows": int(processed["response_valid"].sum()),
    "missing_or_invalid_responses": missing_response_count,
    "responses_at_output_token_limit": token_limit_count,
    "quality_flagged_responses": flagged_response_count,
    "duplicate_prompt_model_runs": duplicate_key_count,
    "complete_valid_groups_of_10": complete_valid_groups,
    "expected_groups_of_10": total_expected_groups,
    "models": sorted(processed["model_name"].unique().tolist()),
    "domains": sorted(processed["domain"].unique().tolist()),
    "collection_max_output_tokens": COLLECTION_MAX_OUTPUT_TOKENS,
    "random_seed": RANDOM_SEED,
}

analysis_path = OUTPUT_DIR / "analysis_ready_responses.csv"
audit_path = OUTPUT_DIR / "processed_responses_audit.csv"
flagged_path = OUTPUT_DIR / "quality_flagged_responses.csv"
excluded_path = OUTPUT_DIR / "excluded_invalid_responses.csv"
checks_path = OUTPUT_DIR / "data_quality_checks.csv"
dictionary_path = OUTPUT_DIR / "data_dictionary.csv"
summary_path = OUTPUT_DIR / "cleaning_summary.json"

processed.to_csv(audit_path, index=False)
analysis_ready.to_csv(analysis_path, index=False)
flagged_responses.to_csv(flagged_path, index=False)
excluded_invalid.to_csv(excluded_path, index=False)
checks.to_csv(checks_path, index=False)
data_dictionary.to_csv(dictionary_path, index=False)
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

package_path = OUTPUT_DIR / "cleaned_data_package.zip"
package_files = [analysis_path, audit_path, flagged_path, excluded_path, checks_path, dictionary_path, summary_path]
with zipfile.ZipFile(package_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in package_files:
        archive.write(path, arcname=path.name)

print(json.dumps(summary, indent=2))
print("\nCreated files:")
for path in package_files + [package_path]:
    print(" -", path)


## Step 10 — Final verification and optional download

The checks below reload the exported files, confirm their row counts and verify that the compact dataset has a unique prompt–model–run key. In Colab, the ZIP can then be downloaded with one click.


In [ ]:
reloaded_analysis = pd.read_csv(analysis_path)
reloaded_audit = pd.read_csv(audit_path)
reloaded_flags = pd.read_csv(flagged_path)

assert len(reloaded_audit) == EXPECTED_LONG_ROWS
expected_analysis_rows = int((
    processed["response_valid"]
    & ~processed.duplicated(["prompt_id", "model_name", "run_number"], keep="first")
).sum())
assert len(reloaded_analysis) == expected_analysis_rows
assert reloaded_analysis.duplicated(["prompt_id", "model_name", "run_number"]).sum() == 0
assert len(reloaded_flags) == flagged_response_count
assert set(reloaded_analysis["model_name"].unique()) == set(MODELS)
assert not reloaded_analysis["response_text"].fillna("").str.strip().eq("").any()

print("Export verification passed.")
print("Dataset status:", dataset_status)
print("ZIP package:", package_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(package_path))


## Interpretation of the status

- **ANALYSIS_READY_COMPLETE:** the collection is fully balanced and has no critical collection flags.
- **ANALYSIS_READY_WITH_LIMITATIONS:** the data are cleaned and may be analysed, but missing and token-limited outputs must be reported and addressed with robustness checks. See `quality_flagged_responses.csv`.

Keep the original `data_collection_all_responses.xlsx` unchanged as the raw audit source.
